# NL2Shell v2 — Fine-tune Qwen3.5-0.8B on 11,894 Deduplicated Pairs

**v2 upgrades over v1:**
- Dataset: `AryaYT/nl2shell-training` (11,894 deduplicated rows, already ChatML formatted — 47% more than v1)
- LoRA rank: r=16, alpha=32 (same as v1 — Amp review: r=32 overfits on 12k examples)
- Epochs: 4 (increased from v1's 3 — more data supports slightly more passes)
- Effective batch: 16 × 4 = 64 (H100 80GB headroom)
- Warmup: 5% of total steps (per Amp recommendation)
- Hardware: H100 HIGHMEM

**Output:** `AryaYT/nl2shell-0.8b` (merged 16-bit + GGUF q4_k_m + q8_0)

In [ ]:
# Cell 1: Install dependencies
import subprocess
import sys

print("[0/6] Installing dependencies...")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    "trl", "peft", "accelerate", "bitsandbytes",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "datasets", "huggingface_hub",
])

print("Dependencies installed.")

In [ ]:
# Cell 2: HF login + imports
import os
import math
import torch
from datasets import load_dataset
from huggingface_hub import login, HfApi, create_repo

os.environ["WANDB_DISABLED"] = "true"

# ── Constants ────────────────────────────────────────────────────────────────
HF_TOKEN       = os.environ.get("HF_TOKEN", "")
MODEL_NAME     = "Qwen/Qwen3.5-0.8B"
DATASET_REPO   = "AryaYT/nl2shell-training"
OUTPUT_REPO    = "AryaYT/nl2shell-0.8b"
MAX_SEQ_LENGTH = 512

# ── Inline eval constants (self-contained — no prepare.py needed) ─────────
SYSTEM_PROMPT = (
    "You are an expert shell programmer. "
    "Given a natural language request, output ONLY the corresponding shell command. "
    "No explanations."
)

EVAL_PROMPTS = [
    "list all files in the current directory",
    "find all Python files larger than 1MB",
    "show the last 20 lines of a log file",
    "create a compressed backup of the home directory",
    "check which process is using port 8080",
    "show all running processes sorted by memory usage",
    "count the number of lines in all .py files recursively",
]

print("[1/6] Logging into HuggingFace...")
login(token=HF_TOKEN)
print("Logged in.")

In [ ]:
# Cell 3: Load dataset from AryaYT/nl2shell-training
# The dataset already has a 'text' column in ChatML format — load and use directly.
print("[2/6] Loading dataset...")

raw = load_dataset(DATASET_REPO, split="train")
print(f"  Loaded {len(raw):,} rows from {DATASET_REPO}")
print(f"  Columns: {raw.column_names}")
print(f"  Sample row:\n{raw[0]['text'][:300]}")

# Keep only the 'text' column; shuffle for good measure
dataset = raw.select_columns(["text"]).shuffle(seed=42)
print(f"  Final training examples: {len(dataset):,}")

In [ ]:
# Cell 4: Load Qwen3.5-0.8B with Unsloth FastLanguageModel (4-bit)
#         Falls back to standard transformers + PEFT if Unsloth unavailable.
print(f"[3/6] Loading {MODEL_NAME}...")

USE_UNSLOTH = True
try:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,          # auto-detect bf16 on H100
        load_in_4bit=True,
    )
    print("  Loaded with Unsloth FastLanguageModel (4-bit QLoRA)")

except Exception as e:
    print(f"  Unsloth failed: {e}")
    print("  Falling back to standard transformers + PEFT...")
    USE_UNSLOTH = False

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("  Loaded with transformers (4-bit)")

In [ ]:
# Cell 5: Apply QLoRA — r=16, alpha=32, all linear layers, dropout=0.05
# NOTE: Amp review recommended r=16 over r=32 to avoid overfitting on 12k examples
print("[4a/6] Applying QLoRA adapters...")

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

if USE_UNSLOTH:
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    print("  Applied via Unsloth FastLanguageModel.get_peft_model")
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("  Applied via PEFT LoraConfig")

model.print_trainable_parameters()
print(f"  r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

In [ ]:
# Cell 6: SFTTrainer — 4 epochs, batch=16, grad_accum=4, lr=2e-4, cosine, bf16
#         packing=True, train_on_responses_only, warmup 5%
from trl import SFTTrainer
from transformers import TrainingArguments

print("[4b/6] Configuring trainer...")

NUM_EPOCHS    = 4
BATCH_SIZE    = 16
GRAD_ACCUM    = 4
EFFECTIVE_BS  = BATCH_SIZE * GRAD_ACCUM   # 64
CHECKPOINT_DIR = "/content/nl2shell-v2-checkpoints"

steps_per_epoch = math.ceil(len(dataset) / EFFECTIVE_BS)
total_steps     = steps_per_epoch * NUM_EPOCHS
save_steps      = max(steps_per_epoch // 2, 50)
warmup_steps    = max(int(total_steps * 0.05), 20)  # 5% warmup (Amp recommendation)

print(f"  Examples:      {len(dataset):,}")
print(f"  Epochs:        {NUM_EPOCHS}")
print(f"  Batch:         {BATCH_SIZE} x {GRAD_ACCUM} = {EFFECTIVE_BS} effective")
print(f"  Steps/epoch:   {steps_per_epoch}")
print(f"  Total steps:   {total_steps}")
print(f"  Warmup steps:  {warmup_steps} (5%)")
print(f"  Save every:    {save_steps} steps")
print(f"  Checkpoint dir: {CHECKPOINT_DIR}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        fp16=False,
        bf16=True,                   # H100 native bf16
        logging_steps=10,
        save_steps=save_steps,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=CHECKPOINT_DIR,
        report_to="none",
        dataloader_num_workers=4,
        group_by_length=False,       # packing=True handles this
    ),
)

# Mask system/user tokens — train only on assistant responses
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("  train_on_responses_only applied")
except Exception as e:
    print(f"  train_on_responses_only skipped: {e}")

print("Starting training...")
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Loss:  {trainer_stats.training_loss:.4f}")
print(f"  Time:  {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"  Steps: {trainer_stats.global_step}")

In [ ]:
# Cell 7: Evaluation — inline from prepare.py (self-contained)
print("[5/6] Running evaluation...")


def run_eval(model, tokenizer):
    """Run 7 test prompts through the model and print NL -> CMD."""
    print("\n" + "=" * 60)
    print("EVALUATION")
    print("=" * 60)

    for prompt in EVAL_PROMPTS:
        chatml_input = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        inputs = tokenizer(chatml_input, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
        # Extract only the assistant turn
        if "<|im_start|>assistant\n" in full_output:
            cmd = full_output.split("<|im_start|>assistant\n")[-1]
            cmd = cmd.split("<|im_end|>")[0].strip()
        else:
            cmd = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1]:],
                skip_special_tokens=True,
            ).strip()

        print(f"  NL:  {prompt}")
        print(f"  CMD: {cmd}")
        print()


# Switch to inference mode for evaluation
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

run_eval(model, tokenizer)

In [ ]:
# Cell 8: Export GGUF — q4_k_m and q8_0 via Unsloth save_pretrained_gguf
print("[6a/6] Exporting GGUF quantizations...")

if USE_UNSLOTH:
    print("  Saving merged 16-bit model...")
    model.save_pretrained_merged(
        "/content/nl2shell-v2-merged", tokenizer, save_method="merged_16bit"
    )
    print("  Merged 16-bit saved.")

    print("  Exporting GGUF q4_k_m...")
    try:
        model.save_pretrained_gguf(
            "/content/nl2shell-v2-gguf-q4",
            tokenizer,
            quantization_method="q4_k_m",
        )
        print("    q4_k_m done")
    except Exception as e:
        print(f"    q4_k_m failed: {e}")

    print("  Exporting GGUF q8_0...")
    try:
        model.save_pretrained_gguf(
            "/content/nl2shell-v2-gguf-q8",
            tokenizer,
            quantization_method="q8_0",
        )
        print("    q8_0 done")
    except Exception as e:
        print(f"    q8_0 failed: {e}")
else:
    print("  Unsloth not available — skipping GGUF export.")
    print("  To export GGUF manually: merge adapter then run llama.cpp quantize.")

In [ ]:
# Cell 9: Push merged model + GGUF to HuggingFace AryaYT/nl2shell-0.8b
print("[6b/6] Pushing to HuggingFace...")

api = HfApi()
create_repo(OUTPUT_REPO, private=False, exist_ok=True)
print(f"  Repo ready: https://huggingface.co/{OUTPUT_REPO}")

if USE_UNSLOTH:
    # Push merged 16-bit model + tokenizer
    model.push_to_hub(OUTPUT_REPO, tokenizer=tokenizer)
    print("  Merged model pushed")

    # Push GGUF files
    for gguf_dir, label in [
        ("/content/nl2shell-v2-gguf-q4", "q4_k_m"),
        ("/content/nl2shell-v2-gguf-q8", "q8_0"),
    ]:
        if os.path.exists(gguf_dir):
            for f in os.listdir(gguf_dir):
                if f.endswith(".gguf"):
                    api.upload_file(
                        path_or_fileobj=os.path.join(gguf_dir, f),
                        path_in_repo=f"gguf/{f}",
                        repo_id=OUTPUT_REPO,
                    )
                    print(f"    Uploaded gguf/{f} ({label})")
        else:
            print(f"  Warning: {gguf_dir} not found, skipping {label}")
else:
    # Standard PEFT fallback: push adapter only
    model.push_to_hub(OUTPUT_REPO)
    tokenizer.push_to_hub(OUTPUT_REPO)
    print("  LoRA adapter pushed (merge manually for GGUF)")

# Upload model card
MODEL_CARD = """---
license: mit
base_model: Qwen/Qwen3.5-0.8B
tags:
  - nl2bash
  - shell
  - terminal
  - command-line
  - qwen3.5
  - qlora
  - cloudagi
datasets:
  - AryaYT/nl2shell-training
language:
  - en
pipeline_tag: text-generation
---

# NL2Shell 0.8B — Natural Language to Shell Commands (v2)

Ultra-lightweight model for converting natural language to Unix/macOS shell commands.
Fine-tuned from Qwen/Qwen3.5-0.8B using QLoRA on 11,894 deduplicated NL-to-shell pairs.

## Quick Start

### Ollama (recommended)
```bash
ollama run hf.co/AryaYT/nl2shell-0.8b
```

### Python
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("AryaYT/nl2shell-0.8b")
tokenizer = AutoTokenizer.from_pretrained("AryaYT/nl2shell-0.8b")

prompt = (
    "<|im_start|>system\\n"
    "You are an expert shell programmer. Given a natural language request, "
    "output ONLY the corresponding shell command. No explanations.<|im_end|>\\n"
    "<|im_start|>user\\nlist all files in the current directory<|im_end|>\\n"
    "<|im_start|>assistant\\n"
)
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))
```

## Training Details

| Parameter | v1 | v2 |
|-----------|----|----|
| Dataset | 8,130 pairs | 11,894 pairs (+47%) |
| LoRA rank / alpha | 16 / 32 | 16 / 32 |
| Epochs | 3 | 4 |
| Effective batch | 32 | 64 |
| Warmup | None | 5% of steps |
| Hardware | A100 40GB | H100 80GB |
| Trainable params | 6.4M (0.74%) | 6.4M (0.74%) |

- **Base model:** [Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B) — hybrid DeltaNet architecture
- **Method:** QLoRA (4-bit NF4, all linear layers, dropout 0.05)
- **Dataset:** [AryaYT/nl2shell-training](https://huggingface.co/datasets/AryaYT/nl2shell-training)
- **Loss masking:** Response-only (train on assistant tokens only)
- **GGUF:** q4_k_m (~400MB edge), q8_0 (~650MB desktop)
- **License:** MIT
- **Built by:** [Arya Teja](https://github.com/aryateja2106) | [CloudAGI](https://cloudagi.ai)
"""

api.upload_file(
    path_or_fileobj=MODEL_CARD.encode(),
    path_in_repo="README.md",
    repo_id=OUTPUT_REPO,
)
print("  Model card uploaded")
print(f"\nModel live at: https://huggingface.co/{OUTPUT_REPO}")
print("TRAINING_COMPLETE")